# 05c — DenseNet-121 pretrained ImageNet (Pendekatan 2)

Muat DenseNet-121 dengan bobot **ImageNet** (`DenseNet121_Weights.IMAGENET1K_V1`),
lalu sesuaikan untuk klasifikasi **7 kelas awan GCD + CCSN**.
Classifier ImageNet diganti dari `Linear(1024, 1000)` menjadi
`Linear(1024, 7)` yang diinisialisasi acak.

Notebook ini memakai **feature extraction**, mengikuti baseline proyek:
parameter backbone dibekukan dan **7.175 parameter classifier** dilatih.
Statistik berjalan BatchNorm tetap beradaptasi pada data awan ketika trainer
memanggil `model.train()`; pembekuan di sini berlaku pada parameter yang
dioptimasi, bukan seluruh state backbone.

Eksperimen: `exp_005c_densenet121_imagenet`.
Konfigurasi: `configs/experiments/exp_005c_densenet121_imagenet.yaml`.
Jalankan sel secara berurutan. Bobot ImageNet dimuat dari cache PyTorch atau
diunduh saat pertama kali dibutuhkan; tidak memakai checkpoint pendekatan 1.

Referensi: `docs/pelatihan/densenet121/BASELINE_DENSENET121.md` dan
`docs/pelatihan/TRAINING_GUIDE.md`.


## 1. Setup & cek perangkat

In [ ]:
import os, sys
from pathlib import Path

# Pastikan berjalan dari akar repo (folder yang memuat src/).
CURRENT = Path.cwd().resolve()

REPO = next(
    (
        folder
        for folder in (CURRENT, *CURRENT.parents)
        if (folder / "src").is_dir()
    ),
    None,
)

if REPO is None:
    raise FileNotFoundError(
        f"Folder 'src' tidak ditemukan dari {CURRENT} "
        "maupun folder induknya. Periksa lokasi proyek."
    )

os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
print("repo   :", REPO)
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available(),
      "(" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "(CPU)")

## 2. Konfigurasi run

- `EPOCHS=None`: memakai **10 epoch** dari konfigurasi eksperimen;
  gunakan `EPOCHS=1` untuk uji singkat.
- `BATCH_SIZE=None`: memakai **8**; turunkan menjadi 4 bila kehabisan memori.
- `NUM_WORKERS=None`: memakai **0**, sesuai konfigurasi Jupyter Windows.
- Optimizer **AdamW**, learning rate **0,0001 (1e-4)**, tanpa scheduler.
- Input **512×512 RGB**, normalisasi **/255 ke [0, 1]**, mengikuti baseline
  proyek dan pendekatan 1. Ini merupakan pilihan praproses proyek;
  standardisasi mean/std ImageNet tidak diterapkan pada eksperimen ini.

Dataset, seed, epoch, batch, optimizer, dan praproses mengikuti pendekatan 1.
Bobot awal dan jumlah parameter yang dioptimasi berbeda, sehingga perbandingan
hasil mencerminkan kedua perbedaan tersebut.

Konfigurasi efektif disimpan sebelum training. Pemilihan checkpoint memakai
**validation macro-F1**; test hanya dievaluasi pada tahap akhir.


In [ ]:
from src.utils.config import load_config, set_seed, resolve_device

CONFIG_PATH = "configs/experiments/exp_005c_densenet121_imagenet.yaml"
EPOCHS = None       # None = config (10); 1 untuk uji cepat
BATCH_SIZE = None   # None = config (8); turunkan ke 4 atau 2 bila OOM
NUM_WORKERS = None # None = config (0)
DEVICE = "auto"    # "auto" | "cpu" | "cuda"

cfg = load_config(CONFIG_PATH)
for key, value in (
    ("epochs", EPOCHS),
    ("batch_size", BATCH_SIZE),
    ("num_workers", NUM_WORKERS),
):
    if value is not None:
        cfg["training"][key] = value

# Pastikan konfigurasi memakai ImageNet dan mode feature extraction.
assert cfg["model"]["architecture"] == "densenet121"
assert cfg["model"]["pretrained"] == "imagenet", "Pendekatan 2 memerlukan bobot ImageNet."
assert cfg["model"]["finetune"] == "feature_extraction", "Notebook ini melatih classifier saja."
assert cfg["model"]["num_classes"] == cfg["data"]["num_classes"]
set_seed(cfg["seed"])
device = resolve_device(DEVICE)
name = cfg["name"]
print(f"eksperimen {name} | device {device} | epochs {cfg['training']['epochs']} "
      f"| batch {cfg['training']['batch_size']} | workers {cfg['training']['num_workers']}")


## 3. Data (train seimbang 21.000 / validation 1.851)

Train menggunakan `combined_balanced` (3.000 citra per kelas).
Validation memakai split gabungan yang sudah tersedia.
Test resmi 9.000 citra hanya dimuat saat evaluasi akhir.


In [ ]:
from src.datasets.cloud_dataset import build_dataloader

train_loader = build_dataloader("train", cfg)
val_loader = build_dataloader("val", cfg)
print("sampel -> train:", len(train_loader.dataset), "| val:", len(val_loader.dataset))
print("batches -> train:", len(train_loader), "| val:", len(val_loader))


## 4. Model: pretrained ImageNet dan classifier 7 kelas


In [ ]:
from src.models.baseline import build_model, count_parameters

# Seed mengontrol inisialisasi classifier baru.
set_seed(cfg["seed"])
model = build_model(
    cfg["model"]["architecture"],
    num_classes=cfg["data"]["num_classes"],
    pretrained=cfg["model"]["pretrained"] == "imagenet",
    finetune=cfg["model"]["finetune"],
)
total, trainable = count_parameters(model)
head_parameters = sum(p.numel() for p in model.classifier.parameters())
assert not any(p.requires_grad for p in model.features.parameters())
assert all(p.requires_grad for p in model.classifier.parameters())
assert trainable == head_parameters == 7175
assert model.classifier.out_features == cfg["data"]["num_classes"]
print(f"DenseNet-121 | ImageNet IMAGENET1K_V1 | total {total:,} | trainable {trainable:,}")
print("classifier:", model.classifier)
print("Parameter backbone dibekukan; statistik BatchNorm tetap beradaptasi saat training.")


## 5. Pelatihan classifier untuk klasifikasi awan

Sel ini melatih classifier baru. Parameter backbone tidak menerima pembaruan
optimizer. Statistik berjalan BatchNorm tetap diperbarui, sesuai perilaku
trainer baseline proyek.

Checkpoint terbaik dipilih dari validation macro-F1 dan disimpan ke
`checkpoints/exp_005c_densenet121_imagenet/best.pt`.
Artefak run disimpan ke `experiments/exp_005c_densenet121_imagenet/`.

Untuk mengulang eksperimen dari bobot **ImageNet** dan classifier baru,
jalankan kembali sel model sebelum sel training. Menjalankan sel training
saja memakai model yang masih berada di memori.


In [ ]:
import json
import yaml
from dataclasses import asdict
import pandas as pd
from src.training.trainer import fit

exp_dir = Path(cfg["output"]["experiments_dir"]) / name
exp_dir.mkdir(parents=True, exist_ok=True)
(exp_dir / "config.yaml").write_text(
    yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8"
)
(exp_dir / "model_architecture.txt").write_text(
    f"{model}\n\nTotal parameters: {total}\nTrainable parameters: {trainable}\n",
    encoding="utf-8",
)
ckpt = Path(cfg["checkpoint"]["dir"]) / name / "best.pt"
result = fit(model, train_loader, val_loader, cfg, device, ckpt, progress=True)

hist = pd.DataFrame([asdict(record) for record in result.history])
hist.to_csv(exp_dir / "training_history.csv", index=False)
(exp_dir / "metrics.json").write_text(
    json.dumps({
        "experiment": name,
        "best_epoch": result.best_epoch,
        "monitor": cfg["checkpoint"]["monitor"],
        "best_val_metric": result.best_metric,
        "test": {"status": "not_evaluated"},
    }, indent=2),
    encoding="utf-8",
)
print(f"SELESAI. best_epoch={result.best_epoch} | best val_macro_f1={result.best_metric:.4f}")
print("checkpoint:", ckpt)
print("artefak:", exp_dir)


## 6. Ringkasan riwayat pelatihan

In [ ]:
from IPython.display import display

display(hist.round(4))
history_plot = hist.set_index("epoch")
history_plot[["train_loss", "val_loss"]].plot(
    title=f"{name}: loss per epoch", figsize=(7, 4)
).set_xlabel("epoch")
history_plot[["val_macro_f1", "val_balanced_accuracy", "val_accuracy"]].plot(
    title=f"{name}: metrik validation per epoch", figsize=(7, 4)
).set_xlabel("epoch")


## 7. Evaluasi test akhir (opsional, sekali setelah model dipilih)

Ubah `EVALUATE_TEST=True` hanya setelah keputusan model dan hyperparameter
selesai berdasarkan validation. Evaluasi memuat **checkpoint terbaik dari
pelatihan ini**, lalu menyimpan metrik, classification report, dan confusion
matrix ke folder eksperimen.


In [ ]:
EVALUATE_TEST = False

if EVALUATE_TEST:
    from src.training.trainer import build_criterion, evaluate
    from src.evaluation.metrics import compute_metrics
    from scripts.train import write_outputs

    state = torch.load(ckpt, map_location=device)
    model.load_state_dict(state["model_state_dict"])
    model.to(device)
    test_loader = build_dataloader("test", cfg)
    _, y_true, y_pred = evaluate(model, test_loader, build_criterion(cfg, device), device)
    test_metrics = compute_metrics(y_true, y_pred, cfg["data"]["num_classes"])
    write_outputs(exp_dir, cfg, result, test_metrics, y_true, y_pred)
    print(f"TEST macro-F1 {test_metrics['macro_f1']:.4f} | "
          f"balanced acc {test_metrics['balanced_accuracy']:.4f} | "
          f"acc {test_metrics['accuracy']:.4f}")
else:
    print("Test belum dievaluasi. Gunakan validation untuk pemilihan model.")
